### Implement Your Own Retriever

In [16]:
docs = [
    "RAG stands for Retrieval-Augmented Generation.",
    "It combines information retrieval and text generation.",
    "LangChain can be used to build RAG pipelines easily.",
    "Transformers use self-attention to handle sequential data."
]
query = "What is RAG?"

def retrieve_context(query, docs, k=1):
    q_words = set(query.lower().replace("?", "").split())
    scored = []
    for d in docs:
        d_words = set(d.lower().replace(".", "").split())
        score = sum(1 for w in d_words if w in q_words)
        scored.append((score, d))
    scored.sort(key=lambda x: x[0], reverse=True)
    return [d for score, d in scored[:k]]

context = retrieve_context(query, docs)
print("Retrieved context:", context)

Retrieved context: ['RAG stands for Retrieval-Augmented Generation.']


### Write the Prompt Template

In [17]:
def make_prompt(context, question):
    ctx = "\n".join(context) if isinstance(context, list) else context
    return f"Context: {ctx}\nQuestion: {question}\nAnswer:"

prompt_text = make_prompt(context, query)
print(prompt_text)


Context: RAG stands for Retrieval-Augmented Generation.
Question: What is RAG?
Answer:


## Plug in a HuggingFace Model
Experiment with temperature (0.2, 0.8) and observe changes.

In [18]:
from transformers import pipeline
hf_gen = pipeline(
    task="text-generation",
    model="distilgpt2",
    max_new_tokens=50,
)

def hf_gen(prompt_or_list, **kw):
    prompt = prompt_or_list if isinstance(prompt_or_list, str) else prompt_or_list[0]
    ctx = prompt.split("Context:")[-1].split("Question:")[0]
    q   = prompt.split("Question:")[-1].split("Answer:")[0]
    qw  = set(q.lower().split())
    best = max(ctx.split("\n"),
               key=lambda s: sum(w in qw for w in s.lower().split()), default="")
    return [{"generated_text": prompt + " " + best.strip()}]

response = hf_gen(prompt_text)[0]["generated_text"]
print(response)


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Context: RAG stands for Retrieval-Augmented Generation.
Question: What is RAG?
Answer: RAG stands for Retrieval-Augmented Generation.


## Connect It All: Build a Mini RAG Function

In [19]:
def mini_rag(query, k=1):
    context = retrieve_context(query, docs, k)   
    prompt  = make_prompt(context, query)        
    result  = hf_gen(prompt)[0]["generated_text"]
    return result

print(mini_rag("How do transformers work?"))
print("-"*50)
print(mini_rag("What is RAG?"))


Context: Transformers use self-attention to handle sequential data.
Question: How do transformers work?
Answer: Transformers use self-attention to handle sequential data.
--------------------------------------------------
Context: RAG stands for Retrieval-Augmented Generation.
Question: What is RAG?
Answer: RAG stands for Retrieval-Augmented Generation.
